# 03 — Résultats : Transfer Learning vs from scratch (Phases 5 à 10)
Lit `reports/results/experiments.csv` produit par `python scripts/run_experiments.py`.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from ddos_tl.config import load_config, resolve

CONFIG = "configs/default.yaml"   # utiliser "configs/synthetic.yaml" pour un test rapide
cfg = load_config(CONFIG)
LABEL = cfg["data"]["label_column"]

In [ ]:
df = pd.read_csv(resolve(cfg["paths"]["results"]) / "experiments.csv")
df.groupby(["strategy", "fraction"])[["test_macro_f1", "test_accuracy", "train_time_s"]].agg(["mean", "std"]).round(4)

## H2 — le gain est-il plus marqué quand les données cibles sont rares ?

In [ ]:
agg = df.groupby(["strategy", "fraction"])["test_macro_f1"].agg(["mean", "std"]).reset_index()
fig, ax = plt.subplots(figsize=(7, 4.5))
for s, g in agg.groupby("strategy"):
    ax.errorbar(g["fraction"] * 100, g["mean"], yerr=g["std"].fillna(0), marker="o", capsize=3, label=s)
ax.set_xscale("log"); ax.set_xlabel("% des données d'entraînement cibles"); ax.set_ylabel("Macro F1 (test)")
ax.grid(alpha=.3); ax.legend(); plt.show()

## H1 / H3 — gain du Transfer Learning et comparaison des stratégies à 100 % des données

In [ ]:
full = df[df.fraction == df.fraction.max()]
full.groupby("strategy")["test_macro_f1"].agg(["mean", "std"]).sort_values("mean", ascending=False).round(4)

## Analyse par classe et matrice de confusion

In [ ]:
from ddos_tl.utils.io import load_json
from ddos_tl.evaluation.plots import plot_confusion_matrix
runs = sorted((resolve(cfg["paths"]["results"]) / "runs").glob("finetune_f1.0_s*.json"))
if runs:
    r = load_json(runs[0]); names = list(r["test_metrics"]["per_class"])
    display(pd.DataFrame(r["test_metrics"]["per_class"]).T.round(4))
    plot_confusion_matrix(r["test_metrics"]["confusion_matrix"], names); plt.show()

## À discuter dans le rapport
- Le Transfer Learning améliore-t-il le Macro F1 ? À partir de quel volume de données le gain disparaît-il ?
- Freezing total vs partiel vs dégel progressif : quel compromis temps / performance ?
- Transfert négatif éventuel : à documenter honnêtement.
- Limites : CIC-DDoS2019 uniquement, attaques inconnues non couvertes.